# Lọc ảnh chất lượng thấp và nhóm không phù hợp

Đã rà đủ 4.357/4.357 ảnh. Theo quyết định người dùng, loại 79 cặp ảnh/nhãn khỏi candidate: 42 ảnh mờ/tối/vỡ hạt và toàn bộ 37 ảnh trong PDF review do lặp/chồng lấp, hình minh họa hoặc đối tượng/box không đạt yêu cầu chất lượng. Còn 4.278 cặp dự kiến: 3.778 train / 299 val / 201 test.

File quyết định phiên bản mới `decisions-full-review-v2-drop-ambiguous.json` đã lưu trong thư mục scan trên Drive và được pin SHA ở code cell. Bản trước giữ riêng để đối chiếu.

**Chạy duy nhất code cell trên Colab CPU** để tạo ZIP candidate trong `BottleQC/datasets/blur-filtered/blur-filtered-b17e060caa43d8c3/`. Cell giữ nguyên ZIP nguồn, loại cả ảnh và nhãn theo cặp, rồi đọc lại hash output. Không cần GPU; đây chỉ là lọc dữ liệu, chưa train.

Val bị loại 8 ảnh; khi so model cần dùng cùng tập val đã lọc cho cả model cũ và mới. `training_ready=False`: việc loại ảnh chưa chứng minh nhãn còn lại đúng và chưa hoàn tất gate dữ liệu v2.


In [1]:
# Lọc ảnh mờ độc lập: scan trên Colab CPU; build chỉ sau khi có quyết định review.
import csv
import hashlib
import io
import json
import math
import sys
import zipfile
from pathlib import Path, PurePosixPath

import numpy as np
from PIL import Image, ImageDraw

BLUR_ACTION = "build"  # Review đủ 4.357 ảnh; tạo candidate riêng, không sửa nguồn.
BLUR_DECISIONS_FILE = "decisions-full-review-v2-drop-ambiguous.json"
BLUR_DECISIONS_SHA = "62d42a5103387231a4bfadc013a08e68281b2c6079bf68efb691c3f5ebcc1eb0"
BLUR_REVIEW_PAGE = 1  # lô xem tiếp: 2, 3, ...; không quét lại điểm ảnh khi đã có COMPLETE
BLUR_SOURCE_REL = "BottleQC/datasets/industrial-watch-v2/20261102T070951Z-85f802/split-candidates/test-val-train-9a23a171beefc8a5/prepared-candidate.zip"
BLUR_SOURCE_SHA = "28ee24280bce8184acd1cdf9a97e850788e908109bfa662efecb07a8a7b179eb"
BLUR_CONFIG = {"revision": 1, "full_long_edge": 640, "bottle_long_edge": 256,
               "min_crop_edge": 48, "review_page_size": 72}

def blur_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def blur_hash(data):
    return hashlib.sha256(data).hexdigest()

def blur_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def blur_score(gray, long_edge):
    scale = min(1.0, long_edge / max(gray.shape))
    if scale < 1:
        gray = np.asarray(Image.fromarray(gray).resize(
            (max(1, round(gray.shape[1] * scale)), max(1, round(gray.shape[0] * scale))),
            Image.Resampling.BOX))
    if min(gray.shape) < 3:
        return 0.0
    gray = gray.astype(np.float64)
    laplacian = (gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2]
                 + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1])
    return float(laplacian.var())

def blur_members(z):
    names = z.namelist()
    if len(names) != len(set(names)):
        raise ValueError("ZIP có tên file trùng.")
    for name in names:
        p = PurePosixPath(name)
        if p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name:
            raise ValueError("Đường dẫn ZIP không an toàn: " + name)

def blur_scan(source, output_root, expected_sha, config):
    if blur_sha(source) != expected_sha:
        raise ValueError("SHA ZIP nguồn không khớp; dừng.")
    binding = {"source_sha256": expected_sha, "config": config}
    key = blur_hash(json.dumps(binding, sort_keys=True).encode())[:16]
    out = Path(output_root) / ("blur-" + key)
    if out.exists():
        marker = json.loads((out / "COMPLETE.json").read_text(encoding="utf-8"))
        if marker["binding"] != binding:
            raise ValueError("Hồ sơ cũ khác input.")
        for name, digest in marker["files"].items():
            if blur_sha(out / name) != digest:
                raise ValueError("Artifact scan đã đổi: " + name)
        return out
    # Giữ thư mục chưa hoàn tất để điều tra nếu bị ngắt; không ghi đè khi chạy lại.
    out.mkdir(parents=True)
    rows = []
    with zipfile.ZipFile(source) as z:
        blur_members(z)
        pairs = json.loads(z.read("retained-checksums.json"))
        seen_images, seen_labels = set(), set()
        split_indices = {"train": 0, "val": 0, "test": 0}
        # Cell 11 đánh ID theo thứ tự retained-checksums.json trong từng split.
        for pair in pairs:
            split = pair["split"]
            if split not in split_indices:
                raise ValueError("Split không hợp lệ.")
            image_name, label_name = pair["image"], pair["label"]
            if image_name in seen_images or label_name in seen_labels:
                raise ValueError("Cặp ảnh/nhãn trùng.")
            seen_images.add(image_name); seen_labels.add(label_name)
            if (PurePosixPath(image_name).parts[:2] != ("images", split)
                    or PurePosixPath(label_name).parts[:2] != ("labels", split)):
                raise ValueError("Path/split không khớp.")
            ib, lb = z.read(image_name), z.read(label_name)
            if blur_hash(ib) != pair["image_sha256"] or blur_hash(lb) != pair["label_sha256"]:
                raise ValueError("SHA ảnh/nhãn không khớp: " + image_name)
            with Image.open(io.BytesIO(ib)) as im:
                if im.getexif().get(274, 1) != 1:
                    raise ValueError("Cần xử lý orientation trước: " + image_name)
                gray = np.array(im.convert("L"))
            h, w = gray.shape
            crop_scores = []
            tiny_bottles = 0
            for line in lb.decode("utf-8-sig").splitlines():
                if not line.strip():
                    continue
                cls, cx, cy, bw, bh = map(float, line.split())
                if cls not in (0, 1, 2) or not all(math.isfinite(v) for v in (cx, cy, bw, bh)):
                    raise ValueError("Nhãn không hợp lệ: " + label_name)
                if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                    raise ValueError("Tọa độ nhãn không hợp lệ: " + label_name)
                if cls != 0:
                    continue
                x1, x2 = max(0, math.floor((cx-bw/2)*w)), min(w, math.ceil((cx+bw/2)*w))
                y1, y2 = max(0, math.floor((cy-bh/2)*h)), min(h, math.ceil((cy+bh/2)*h))
                if min(x2-x1, y2-y1) < config["min_crop_edge"]:
                    tiny_bottles += 1
                else:
                    crop_scores.append(blur_score(gray[y1:y2, x1:x2], config["bottle_long_edge"]))
            split_indices[split] += 1
            row = dict(pair, id=f"{split}-{split_indices[split]:06d}", width=w, height=h,
                       full_score=blur_score(gray, config["full_long_edge"]),
                       bottle_min_score=min(crop_scores) if crop_scores else None,
                       tiny_bottle_boxes=tiny_bottles)
            rows.append(row)
            if len(rows) % 250 == 0:
                print("BLUR_SCAN_PROGRESS:", len(rows), "/", len(pairs), flush=True)
        expected = {p[k] for p in pairs for k in ("image", "label")}
        actual = {n for n in z.namelist() if n.startswith(("images/", "labels/")) and not n.endswith("/")}
        if actual != expected:
            raise ValueError("Có ảnh/nhãn ngoài manifest.")
        # Hai bảng xếp hạng, không dùng một ngưỡng Laplacian để tự xóa ảnh.
        train = [r for r in rows if r["split"] == "train"]
        full_order = sorted(train, key=lambda r: (r["full_score"], r["image"]))
        crop_order = sorted((r for r in train if r["bottle_min_score"] is not None),
                            key=lambda r: (r["bottle_min_score"], r["image"]))
        ranked, used = [], set()
        for i in range(len(train)):
            for ranking in (full_order, crop_order):
                if i < len(ranking) and ranking[i]["id"] not in used:
                    ranked.append(ranking[i]); used.add(ranking[i]["id"])
        page = ranked[:config["review_page_size"]]
        previews = out / "review-page-001"
        previews.mkdir()
        for r in page:
            (previews / (r["id"] + "-source" + Path(r["image"]).suffix)).write_bytes(z.read(r["image"]))
        for page_index in range(0, len(page), 6):
            sheet = Image.new("RGB", (960, 700), "white")
            draw = ImageDraw.Draw(sheet)
            for j, r in enumerate(page[page_index:page_index+6]):
                x, y = (j % 3)*320, (j // 3)*350
                with Image.open(io.BytesIO(z.read(r["image"]))) as im:
                    tile = im.convert("RGB"); tile.thumbnail((310, 310))
                    sheet.paste(tile, (x+(320-tile.width)//2, y+30))
                draw.text((x+5, y+3), r["id"] + " full=" + str(round(r["full_score"], 1)), fill="black")
            sheet.save(previews / f"sheet-{page_index//6+1:02d}.jpg", quality=92)
    blur_json(out / "scores.json", rows)
    with (out / "scores.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        fields = ["id", "split", "image", "width", "height", "full_score", "bottle_min_score", "tiny_bottle_boxes"]
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader(); writer.writerows(rows)
    blur_json(out / "review-order.json", [r["id"] for r in ranked])
    blur_json(out / "decisions-template.json", {"source_sha256": expected_sha, "decisions": [
        {"id": r["id"], "image_sha256": r["image_sha256"], "label_sha256": r["label_sha256"],
         "action": "unreviewed", "reason": "", "reviewer_id": "", "reviewed_at_utc": ""} for r in page]})
    files = {p.relative_to(out).as_posix(): blur_sha(p) for p in out.rglob("*") if p.is_file()}
    blur_json(out / "COMPLETE.json", {"binding": binding, "pair_counts": split_indices,
              "files": files, "auto_excluded": 0, "training_ready": False})
    return out

def blur_review_page(source, scan, page_number):
    if type(page_number) is not int or page_number < 1:
        raise ValueError("Số lô review phải là số nguyên >= 1.")
    if page_number == 1:
        return scan / "review-page-001", scan / "decisions-template.json"
    rows = {r["id"]: r for r in json.loads((scan / "scores.json").read_text(encoding="utf-8"))}
    order = json.loads((scan / "review-order.json").read_text(encoding="utf-8"))
    binding = json.loads((scan / "COMPLETE.json").read_text(encoding="utf-8"))["binding"]
    n = binding["config"]["review_page_size"]
    chosen = [rows[i] for i in order[(page_number-1)*n:page_number*n]]
    if not chosen:
        raise ValueError("Đã hết danh sách train để review.")
    out = scan / f"review-page-{page_number:03d}"
    if out.exists():
        marker = json.loads((out / "COMPLETE.json").read_text(encoding="utf-8"))
        if marker["selected_ids"] != [r["id"] for r in chosen]:
            raise ValueError("Lô review khác input.")
        for name, digest in marker["files"].items():
            if blur_sha(out / name) != digest:
                raise ValueError("Lô review đã đổi.")
        return out, out / "decisions-template.json"
    out.mkdir()
    with zipfile.ZipFile(source) as z:
        for start in range(0, len(chosen), 6):
            sheet = Image.new("RGB", (960, 700), "white")
            draw = ImageDraw.Draw(sheet)
            for j, r in enumerate(chosen[start:start+6]):
                raw = z.read(r["image"])
                if blur_hash(raw) != r["image_sha256"]:
                    raise ValueError("Ảnh khác scan.")
                (out / (r["id"]+"-source"+Path(r["image"]).suffix)).write_bytes(raw)
                with Image.open(io.BytesIO(raw)) as im:
                    tile = im.convert("RGB"); tile.thumbnail((310, 310))
                    x, y = (j % 3)*320, (j // 3)*350
                    sheet.paste(tile, (x+(320-tile.width)//2, y+30))
                    draw.text((x+5, y+3), r["id"], fill="black")
            sheet.save(out / f"sheet-{start//6+1:02d}.jpg", quality=92)
    blur_json(out / "decisions-template.json", {"source_sha256": binding["source_sha256"], "decisions": [
        {"id": r["id"], "image_sha256": r["image_sha256"], "label_sha256": r["label_sha256"],
         "action": "unreviewed", "reason": "", "reviewer_id": "", "reviewed_at_utc": ""} for r in chosen]})
    files = {p.name: blur_sha(p) for p in out.iterdir() if p.is_file()}
    blur_json(out / "COMPLETE.json", {"selected_ids": [r["id"] for r in chosen], "files": files})
    return out, out / "decisions-template.json"

def blur_build(source, scan_dir, decision_path, output_root, expected_sha):
    if blur_sha(source) != expected_sha:
        raise ValueError("SHA ZIP nguồn không khớp.")
    scan_dir = Path(scan_dir)
    marker = json.loads((scan_dir / "COMPLETE.json").read_text(encoding="utf-8"))
    if marker["binding"]["source_sha256"] != expected_sha:
        raise ValueError("Scan khác nguồn.")
    for name, digest in marker["files"].items():
        if blur_sha(scan_dir / name) != digest:
            raise ValueError("Scan artifact đã đổi: " + name)
    records = json.loads((scan_dir / "scores.json").read_text(encoding="utf-8"))
    by_id = {r["id"]: r for r in records}
    decision_bytes = Path(decision_path).read_bytes()
    reviewed = json.loads(decision_bytes)
    if reviewed["source_sha256"] != expected_sha:
        raise ValueError("Quyết định khác nguồn.")
    excluded, seen = {}, set()
    for d in reviewed["decisions"]:
        ident = d["id"]
        if ident in seen or ident not in by_id:
            raise ValueError("ID trùng/không tồn tại.")
        seen.add(ident); r = by_id[ident]
        if d["image_sha256"] != r["image_sha256"] or d["label_sha256"] != r["label_sha256"]:
            raise ValueError("Quyết định không khớp byte nguồn.")
        if d["action"] not in {"keep", "exclude_quality", "unresolved"}:
            raise ValueError("Action không hợp lệ.")
        if d["action"] in {"keep", "exclude_quality", "unresolved"}:
            if not all(isinstance(d.get(k), str) and d[k].strip() for k in ("reason", "reviewer_id", "reviewed_at_utc")):
                raise ValueError("Cần lý do và người/thời điểm review.")
        if d["action"] == "exclude_quality":
            if not isinstance(d.get("issue"), str) or not d["issue"].strip():
                raise ValueError("Ảnh loại cần loại lỗi chất lượng.")
            excluded[ident] = d
    if seen != set(by_id):
        raise ValueError("Chưa review đủ toàn bộ ảnh; không build từ lô nhỏ.")
    if not excluded:
        raise ValueError("Chưa có ảnh xác nhận exclude_quality; không tạo candidate rỗng thay đổi.")
    kept = [r for r in records if r["id"] not in excluded]
    counts = {s: sum(r["split"] == s for r in kept) for s in ("train", "val", "test")}
    if not all(counts.values()):
        raise ValueError("Không được làm rỗng split.")
    key = blur_hash((expected_sha + blur_hash(decision_bytes)).encode())[:16]
    out = Path(output_root) / ("blur-filtered-" + key)
    if out.exists():
        raise FileExistsError("Candidate đã tồn tại; kiểm tra COMPLETE, không ghi đè.")
    out.mkdir(parents=True)
    target = out / "prepared-candidate.zip"
    with zipfile.ZipFile(source) as src, zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as dst:
        blur_members(src)
        original_pairs = {p["image"]: p for p in json.loads(src.read("retained-checksums.json"))}
        pairs = []
        for r in kept:
            for name, digest in ((r["image"], r["image_sha256"]), (r["label"], r["label_sha256"])):
                raw = src.read(name)
                if blur_hash(raw) != digest:
                    raise ValueError("Nguồn thay đổi.")
                dst.writestr(name, raw)
            pairs.append(original_pairs[r["image"]])
        dst.writestr("retained-checksums.json", json.dumps(pairs, ensure_ascii=False, indent=2))
        dst.writestr("data.yaml", "train: images/train\nval: images/val\ntest: images/test\nnames:\n  0: bottle\n  1: cap\n  2: label\n")
    # Đọc lại byte trong ZIP output, không chỉ kiểm tra số file.
    with zipfile.ZipFile(target) as check:
        for p in pairs:
            for field in ("image", "label"):
                if blur_hash(check.read(p[field])) != p[field + "_sha256"]:
                    raise ValueError("Readback candidate không khớp.")
    (out / "decisions-reviewed.json").write_bytes(decision_bytes)
    blur_json(out / "COMPLETE.json", {"source_sha256": expected_sha, "candidate_sha256": blur_sha(target),
              "decisions_sha256": blur_hash(decision_bytes), "excluded": list(excluded.values()),
              "counts": counts, "training_ready": False,
              "reviewed_count": len(seen),
              "unresolved_ids": [d["id"] for d in reviewed["decisions"] if d["action"] == "unresolved"],
              "evaluation_split_changed": any(by_id[i]["split"] != "train" for i in excluded),
              "evaluation_policy": "Nếu đổi val/test, đánh giá cả model cũ và mới trên cùng holdout đã lọc; không so trực tiếp với metric cũ.",
              "remaining_work": "Kiểm tra annotation và hoàn tất các gate chất lượng candidate v2; quyết định loại ảnh không chứng minh nhãn còn lại đúng."})
    return out

def run_blur_filter():
    if sys.platform != "linux" or not Path("/content").is_dir():
        raise RuntimeError("Chạy cell này trên Colab CPU, không chạy dataset trên Windows.")
    from google.colab import drive
    root = Path("/content/drive/MyDrive")
    if not root.is_dir():
        drive.mount("/content/drive")
    source = root / BLUR_SOURCE_REL
    print("BLUR_ENV: Colab; thuật toán dùng CPU. Chọn runtime CPU và kiểm tra tài khoản mount Drive.")
    print("BLUR_SOURCE:", source)
    scan = blur_scan(source, root / "BottleQC/datasets/blur-audits", BLUR_SOURCE_SHA, BLUR_CONFIG)
    print("BLUR_SCAN_SAVED:", scan)
    if BLUR_ACTION == "scan":
        from IPython.display import display
        page, template = blur_review_page(source, scan, BLUR_REVIEW_PAGE)
        print("BLUR_REVIEW_TEMPLATE:", template)
        for path in sorted(page.glob("sheet-*.jpg")):
            with Image.open(path) as im:
                display(im.copy())
        print("BLUR_NEXT: xem ảnh gốc trong", page, "; chưa loại ảnh nào. Không chạy train.")
    elif BLUR_ACTION == "build":
        decisions = scan / BLUR_DECISIONS_FILE
        if blur_sha(decisions) != BLUR_DECISIONS_SHA:
            raise ValueError("File quyết định toàn bộ ảnh không đúng SHA đã review.")
        output = blur_build(source, scan, decisions,
                            root / "BottleQC/datasets/blur-filtered", BLUR_SOURCE_SHA)
        summary = json.loads((output / "COMPLETE.json").read_text(encoding="utf-8"))
        print("BLUR_CANDIDATE_SAVED:", output)
        print("BLUR_EXCLUDED_COUNT:", len(summary["excluded"]))
        print("BLUR_COUNTS:", json.dumps(summary["counts"]))
        print("BLUR_UNRESOLVED_COUNT:", len(summary["unresolved_ids"]))
        print("BLUR_EVALUATION_SPLIT_CHANGED:", summary["evaluation_split_changed"])
        print("BLUR_TRAINING_READY:", summary["training_ready"])
    else:
        raise ValueError("BLUR_ACTION chỉ nhận scan hoặc build.")

run_blur_filter()


Mounted at /content/drive
BLUR_ENV: Colab; thuật toán dùng CPU. Chọn runtime CPU và kiểm tra tài khoản mount Drive.
BLUR_SOURCE: /content/drive/MyDrive/BottleQC/datasets/industrial-watch-v2/20261102T070951Z-85f802/split-candidates/test-val-train-9a23a171beefc8a5/prepared-candidate.zip
BLUR_SCAN_SAVED: /content/drive/MyDrive/BottleQC/datasets/blur-audits/blur-322304bcd2bf6857
BLUR_CANDIDATE_SAVED: /content/drive/MyDrive/BottleQC/datasets/blur-filtered/blur-filtered-b17e060caa43d8c3
BLUR_EXCLUDED_COUNT: 79
BLUR_COUNTS: {"train": 3778, "val": 299, "test": 201}
BLUR_UNRESOLVED_COUNT: 0
BLUR_EVALUATION_SPLIT_CHANGED: True
BLUR_TRAINING_READY: False
